# 05 – Chốt ngưỡng cảnh báo sản phẩm (FR-21) và kiểm tra ngược (backtest)
Buổi 5 – Thiết kế chi tiết. Người thực hiện chính: Nguyễn Hoàng Thuận.
FR-21 trong SRS để ngưỡng "chốt sau EDA". Notebook này: (1) xem phân bố Δ điểm 3 tháng và tỷ lệ tiêu cực
của các sản phẩm đủ dữ liệu; (2) chọn ngưỡng theo phân vị; (3) kiểm tra ngược xem nhóm "Cao" có thật sự
tiếp tục kém hơn ở 3 tháng sau hay không.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from backend.core import config as C
from backend.services.alert_service import compute_risk
import vizstyle as V
V.apply()
FIG = ROOT / "notebooks" / "figures"; OUT = ROOT / "notebooks" / "outputs"
K = {}
df = pd.read_csv(C.DATA_PROCESSED / "reviews_clean.csv", usecols=["product_id", "score", "sentiment", "year_month"])

## 1. Phân bố tín hiệu trên các sản phẩm đủ dữ liệu (≥ 10 đánh giá mỗi cửa sổ), tính đến 09/2012

In [2]:
r = compute_risk(df, "2012-09"); e = r[r.risk_level != "insufficient"]
K["eligible"] = len(e); K["active_products_3m"] = int((r.n_3m > 0).sum())
K["delta_pct"] = e.delta.quantile([.05, .1, .25, .5, .75, .9]).round(3).to_dict()
K["neg_pct"] = e.neg_rate_3m.quantile([.5, .75, .9, .95]).round(3).to_dict()
K["overall_neg_rate_3m"] = round(float((df[df.year_month.between("2012-07", "2012-09")].sentiment == "neg").mean()), 4)
K["levels_2012_09"] = e.risk_level.value_counts().to_dict()
K["delta_pct"], K["neg_pct"], K["levels_2012_09"], K["overall_neg_rate_3m"]

({0.05: -0.723, 0.1: -0.528, 0.25: -0.27, 0.5: 0.0, 0.75: 0.227, 0.9: 0.51},
 {0.5: 0.105, 0.75: 0.217, 0.9: 0.412, 0.95: 0.5},
 {'low': 135, 'high': 39, 'medium': 37},
 0.1604)

In [3]:
r.to_csv(C.DATA_PROCESSED / "product_risk.csv", index=False)
top = e.sort_values(["risk_level", "delta"], key=lambda s: s.map({"high": 0, "medium": 1, "low": 2}) if s.name == "risk_level" else s).head(8)
K["top_high"] = top[["product_id", "n_3m", "avg_3m", "neg_rate_3m", "n_prev_3m", "avg_prev_3m", "delta", "risk_level", "reason"]].round(3).to_dict("records")
pd.DataFrame(K["top_high"])

,product_id,n_3m,avg_3m,neg_rate_3m,n_prev_3m,avg_prev_3m,delta,risk_level,reason
0,B00507A02Q,60,2.317,0.617,13,3.846,-1.529,high,"điểm TB 3 tháng giảm 1,53 sao; tỷ lệ tiêu cực 62%"
1,B0031U9VI8,13,2.769,0.538,14,3.929,-1.159,high,"điểm TB 3 tháng giảm 1,16 sao; tỷ lệ tiêu cực 54%"
2,B0012NUVN0,20,2.600,0.500,17,3.706,-1.106,high,"điểm TB 3 tháng giảm 1,11 sao; tỷ lệ tiêu cực 50%"
3,B000WV153I,14,3.857,0.143,10,4.900,-1.043,high,"điểm TB 3 tháng giảm 1,04 sao"
4,B000E243RA,13,3.000,0.462,10,4.000,-1.000,high,"điểm TB 3 tháng giảm 1,00 sao; tỷ lệ tiêu cực 46%"
5,B004WTHCO2,15,3.533,0.267,17,4.471,-0.937,high,"điểm TB 3 tháng giảm 0,94 sao; tỷ lệ tiêu cực 27%"
6,B001T4UMM2,13,4.077,0.231,10,5.000,-0.923,high,"điểm TB 3 tháng giảm 0,92 sao"
7,B007PA32L2,17,3.412,0.353,11,4.273,-0.861,high,"điểm TB 3 tháng giảm 0,86 sao; tỷ lệ tiêu cực 35%"


In [4]:
fig, ax = plt.subplots(figsize=(6.6, 3.4))
col = e.risk_level.map({"high": V.RED, "medium": V.YELLOW, "low": V.BLUE})
ax.scatter(e.delta, e.neg_rate_3m * 100, s=np.clip(e.n_3m, 10, 80), c=col, alpha=0.75, edgecolor="white", linewidth=0.6)
ax.axvline(C.RISK_HIGH_DELTA, color=V.RED, ls="--", lw=1); ax.axvline(C.RISK_MID_DELTA, color=V.YELLOW, ls="--", lw=1)
ax.axhline(C.RISK_HIGH_NEG * 100, color=V.RED, ls="--", lw=1); ax.axhline(C.RISK_MID_NEG * 100, color=V.YELLOW, ls="--", lw=1)
ax.set_xlabel("Δ điểm TB: 3 tháng gần nhất − 3 tháng trước (sao)"); ax.set_ylabel("Tỷ lệ tiêu cực 3 tháng gần nhất (%)")
ax.set_title(f"Sản phẩm đủ dữ liệu tại 09/2012 (n = {len(e)}); kích thước điểm = số đánh giá")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=c, label=l) for c, l in [(V.RED, "Cao"), (V.YELLOW, "Trung bình"), (V.BLUE, "Thấp")]],
          loc="upper right", title="Mức rủi ro", title_fontsize=8)
V.save(fig, FIG / "risk_01_scatter.png")

## 2. Kiểm tra ngược: gắn mức rủi ro tại một tháng, xem 3 tháng tiếp theo

In [5]:
bt = []
for asof, nxt in [("2012-03", ("2012-04", "2012-06")), ("2012-06", ("2012-07", "2012-09"))]:
    rr = compute_risk(df, asof); rr = rr[rr.risk_level != "insufficient"].set_index("product_id")
    f = df[df.year_month.between(*nxt)].groupby("product_id").agg(n_next=("score", "size"), avg_next=("score", "mean"),
                                                                   neg_next=("sentiment", lambda s: (s == "neg").mean()))
    j = rr.join(f, how="inner"); j = j[j.n_next >= 5]
    g = j.groupby("risk_level").agg(san_pham=("n_next", "size"), neg_now=("neg_rate_3m", "mean"), neg_next=("neg_next", "mean"),
                                    avg_now=("avg_3m", "mean"), avg_next=("avg_next", "mean")).reindex(["high", "medium", "low"])
    g["as_of"] = asof; bt.append(g.reset_index())
bt = pd.concat(bt); bt.round(3).to_csv(OUT / "risk_backtest.csv", index=False)
K["backtest"] = bt.round(3).to_dict("records")
bt.round(3)

,risk_level,san_pham,neg_now,neg_next,avg_now,avg_next,as_of
0,high,29,0.360,0.296,3.343,3.575,2012-03
1,medium,37,0.228,0.179,3.817,4.030,2012-03
2,low,109,0.082,0.134,4.423,4.249,2012-03
0,high,31,0.348,0.253,3.434,3.805,2012-06
1,medium,37,0.213,0.218,3.924,3.923,2012-06
2,low,118,0.091,0.110,4.390,4.307,2012-06


In [6]:
(OUT / "risk_results.json").write_text(json.dumps(K, ensure_ascii=False, indent=1, default=float), encoding="utf-8")
print(json.dumps(K, ensure_ascii=False, indent=1, default=float)[:3000])

{
 "eligible": 211,
 "active_products_3m": 21239,
 "delta_pct": {
  "0.05": -0.723,
  "0.1": -0.528,
  "0.25": -0.27,
  "0.5": 0.0,
  "0.75": 0.227,
  "0.9": 0.51
 },
 "neg_pct": {
  "0.5": 0.105,
  "0.75": 0.217,
  "0.9": 0.412,
  "0.95": 0.5
 },
 "overall_neg_rate_3m": 0.1604,
 "levels_2012_09": {
  "low": 135,
  "high": 39,
  "medium": 37
 },
 "top_high": [
  {
   "product_id": "B00507A02Q",
   "n_3m": 60,
   "avg_3m": 2.317,
   "neg_rate_3m": 0.617,
   "n_prev_3m": 13,
   "avg_prev_3m": 3.846,
   "delta": -1.529,
   "risk_level": "high",
   "reason": "điểm TB 3 tháng giảm 1,53 sao; tỷ lệ tiêu cực 62%"
  },
  {
   "product_id": "B0031U9VI8",
   "n_3m": 13,
   "avg_3m": 2.769,
   "neg_rate_3m": 0.538,
   "n_prev_3m": 14,
   "avg_prev_3m": 3.929,
   "delta": -1.159,
   "risk_level": "high",
   "reason": "điểm TB 3 tháng giảm 1,16 sao; tỷ lệ tiêu cực 54%"
  },
  {
   "product_id": "B0012NUVN0",
   "n_3m": 20,
   "avg_3m": 2.6,
   "neg_rate_3m": 0.5,
   "n_prev_3m": 17,
   "avg_prev_3m"